# Qwen3-TTS 0.6B x Sheng (Swahili + English)

Single-speaker fine-tune of `Qwen/Qwen3-TTS-12Hz-0.6B-Base` on
`lyle-mlengineer/kenyan-celebs` - 42 clips / 21 min of one Nairobi speaker
talking Sheng.

**Runtime:** `Runtime -> Change runtime type -> GPU`. L4 (24 GB) is the best
Colab GPU for this; T4 (16 GB) works but needs `--allow-small-gpu`.

Everything is resumable: if the VM dies, re-run the cells from
*"Install"* onwards and the driver continues from the last checkpoint on Drive.

In [ ]:
!nvidia-smi

## 1. Mount Drive (checkpoints, caches and the exported model live there)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Get this project

Set `REPO_URL` to your fork/clone of `audio-speech-ai` (or upload the folder).

In [ ]:
REPO_URL = "https://github.com/<you>/audio-speech-ai.git"   # <-- edit me
REPO_DIR = "/content/audio-speech-ai"

import os
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 "$REPO_URL" "$REPO_DIR"
%cd {REPO_DIR}/finetune-qwentts
!git -C "$REPO_DIR" pull --ff-only || true
!ls

## 3. Install

`qwen-tts` pins `transformers==4.57.3`; let pip do it (a newer transformers is
the most common cause of "AutoModel returned the wrong class" errors).

In [ ]:
!pip install -q -e .
!pip install -q -U qwen-tts
!sheng-qwentts setup

## 4. Pick a profile

| profile | GPU | batch x accum | dtype | attention |
|---|---|---|---|---|
| `l4` | Colab L4 24 GB | 2 x 2 | bfloat16 | sdpa |
| `a100` | Colab A100 | 4 x 1 | bfloat16 | flash_attention_2 |
| `t4` | Colab T4 16 GB | 1 x 4 | float16 | sdpa |

`--profile` can be omitted: the driver detects the GPU. T4 needs
`--allow-small-gpu` (it is a smoke-test profile on purpose - fp16 on Turing is
slower and less stable than bf16).

## 5. Run everything

`data -> model -> codes -> train -> eval -> export`. Re-running with `--resume`
skips finished stages and continues training from `latest.txt`.

`--max-hours 8` stops training cleanly (and checkpoints) before Colab's
disconnect timer, so nothing is lost.

In [ ]:
!sheng-qwentts all --resume --profile l4 --max-hours 8

### ...or drive the stages one at a time

```bash
!sheng-qwentts data                     # 24 kHz wavs + manifests (2 min)
!sheng-qwentts model                    # 0.6B Base + 12 Hz tokenizer (~1.1 GB)
!sheng-qwentts codes                    # audio_codes (GPU, ~1 min)
!sheng-qwentts train --profile l4 --steps 240 --lr 2e-6
!sheng-qwentts eval --compare-base      # A/B against the untouched Base model
!sheng-qwentts export
```

In [ ]:
!sheng-qwentts status

## 6. Listen

The eval stage writes `ft_*.wav` (fine-tuned voice), and with `--compare-base`
also `base_clone_*.wav` (zero-shot clone with the Base model).

In [ ]:
import glob
from IPython.display import Audio, display

for path in sorted(glob.glob('/content/drive/MyDrive/qwentts-sheng/evals/*/*/*.wav'))[:6]:
    print(path)
    display(Audio(path))

## 7. Say something of your own

Sheng = Swahili grammar with English words. Keep the spelling close to the
dataset (Swahili orthography); all-caps slang is read out as if it were English.

In [ ]:
import torch, soundfile as sf, glob
from qwen_tts import Qwen3TTSModel

ckpt = sorted(glob.glob('/content/drive/MyDrive/qwentts-sheng/checkpoints/sheng_06b/step_*'))[-1]
tts = Qwen3TTSModel.from_pretrained(ckpt, device_map="cuda:0", dtype=torch.bfloat16,
                                    attn_implementation="sdpa")

wavs, sr = tts.generate_custom_voice(
    text="Mambo vipi mazee, leo niko fiti kabisa. Wacha tuendelee na stori.",
    speaker="abel",
    language="auto",
    max_new_tokens=1024,
)
sf.write("sample.wav", wavs[0], sr)
Audio("sample.wav")

## 8. Export

`export` bundles a self-contained model dir (config with the `abel` speaker id,
weights, speech tokenizer), the eval samples and a model card under
`/content/drive/MyDrive/qwentts-sheng/exports/sheng_06b`.

In [ ]:
!sheng-qwentts export
!du -sh /content/drive/MyDrive/qwentts-sheng/exports/sheng_06b/*